# 02 - Data Audit

## Objective

Validate the raw datasets and identify any data-quality or business-rule issues that could affect the onboarding funnel, campaign analysis, activation analysis, or airport supply analysis.

In [1]:
import pandas as pd

In [2]:
captains = pd.read_csv("../data/captains.csv")
doc_events = pd.read_csv("../data/doc_events.csv")
approvals = pd.read_csv("../data/approvals.csv")
activation = pd.read_csv("../data/activation.csv")
nudges = pd.read_csv("../data/nudges.csv")
airport_hourly = pd.read_csv("../data/airport_hourly.csv")
airport_trips = pd.read_csv("../data/airport_trips.csv")

In [3]:
datasets = {
    "captains": captains,
    "doc_events": doc_events,
    "approvals": approvals,
    "activation": activation,
    "nudges": nudges,
    "airport_hourly": airport_hourly,
    "airport_trips": airport_trips
}

In [4]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

captains: (25000, 9)
doc_events: (186282, 7)
approvals: (25000, 5)
activation: (4206, 5)
nudges: (16314, 6)
airport_hourly: (10248, 9)
airport_trips: (60000, 9)


## 1. Document Event Lifecycle Audit

Before using document events to construct the onboarding funnel, I want to check whether the event history is chronologically and logically consistent with the onboarding process.

In [5]:
doc_events["event_ts"] = pd.to_datetime(doc_events["event_ts"])
captains["signup_ts"] = pd.to_datetime(captains["signup_ts"])

In [6]:
doc_with_signup = doc_events.merge(
    captains[["captain_id", "signup_ts"]],
    on="captain_id",
    how="left"
)

events_before_signup = doc_with_signup[
    doc_with_signup["event_ts"] < doc_with_signup["signup_ts"]
]

print("Events before signup:", len(events_before_signup))

Events before signup: 0


In [13]:
attempt_check = (
    doc_events
    .groupby(["captain_id", "doc_type"])["attempt_no"]
    .agg(["min", "max", "nunique"])
    .reset_index()
)

display(attempt_check.head(20))

,captain_id,doc_type,min,max,nunique
0,CPT100000,AADHAAR,1,1,1
1,CPT100000,DL,1,1,1
2,CPT100000,FITNESS,1,2,2
3,CPT100000,PERMIT,1,1,1
4,CPT100000,RC,1,1,1
5,CPT100001,DL,1,1,1
6,CPT100001,RC,1,1,1
7,CPT100002,AADHAAR,1,1,1
8,CPT100002,DL,1,1,1
9,CPT100002,FITNESS,1,2,2


In [8]:
attempt_sequences = (
    doc_events
    .groupby(["captain_id", "doc_type"])["attempt_no"]
    .apply(lambda x: sorted(x.unique()))
    .reset_index(name="attempt_sequence")
)

attempt_sequences["has_gap"] = attempt_sequences["attempt_sequence"].apply(
    lambda x: x != list(range(min(x), max(x) + 1))
)

attempt_sequences[attempt_sequences["has_gap"]].head(20)

,captain_id,doc_type,attempt_sequence,has_gap


In [9]:
print(
    "Captain-document combinations with attempt gaps:",
    attempt_sequences["has_gap"].sum()
)

Captain-document combinations with attempt gaps: 0


In [10]:
attempt_event_summary = (
    doc_events
    .groupby(["doc_type", "attempt_no", "event_type"])
    .size()
    .reset_index(name="event_count")
    .sort_values(["doc_type", "attempt_no", "event_type"])
)

attempt_event_summary

,doc_type,attempt_no,event_type,event_count
0,AADHAAR,1,upload_success,14665
1,AADHAAR,1,verification_fail,1439
2,AADHAAR,1,verification_pass,13201
3,AADHAAR,2,upload_success,899
4,AADHAAR,2,verification_fail,11
5,AADHAAR,2,verification_pass,886
6,AADHAAR,3,upload_success,8
7,AADHAAR,3,verification_pass,8
8,DL,1,upload_success,23303
9,DL,1,verification_fail,3308


In [14]:
attempt_outcomes = (
    doc_events
    .groupby(["captain_id", "doc_type", "attempt_no"])["event_type"]
    .agg(lambda x: sorted(x.unique()))
    .reset_index(name="events")
)

display(attempt_outcomes.head(20))

,captain_id,doc_type,attempt_no,events
0,CPT100000,AADHAAR,1,"[upload_success, verification_pass]"
1,CPT100000,DL,1,"[upload_success, verification_pass]"
2,CPT100000,FITNESS,1,"[upload_success, verification_fail]"
3,CPT100000,FITNESS,2,"[upload_success, verification_fail]"
4,CPT100000,PERMIT,1,"[upload_success, verification_pass]"
5,CPT100000,RC,1,"[upload_success, verification_pass]"
6,CPT100001,DL,1,"[upload_success, verification_pass]"
7,CPT100001,RC,1,"[upload_success, verification_fail]"
8,CPT100002,AADHAAR,1,"[upload_success, verification_pass]"
9,CPT100002,DL,1,"[upload_success, verification_pass]"


In [12]:
contradictory_attempts = attempt_outcomes[
    attempt_outcomes["events"].apply(
        lambda x: "verification_pass" in x and "verification_fail" in x
    )
]

print(
    "Attempts containing both verification pass and fail:",
    len(contradictory_attempts)
)

Attempts containing both verification pass and fail: 0


## 2. Document Sequence Audit

The onboarding process has a defined document sequence. I want to check whether the observed document events follow that sequence for each captain.

Permit is only expected for Auto and Cab, so it will be treated separately when checking the expected sequence.

In [15]:
document_order = {
    "DL": 1,
    "RC": 2,
    "AADHAAR": 3,
    "PERMIT": 4,
    "FITNESS": 5,
    "INSURANCE": 6
}

doc_sequence = doc_events.copy()

doc_sequence["document_order"] = doc_sequence["doc_type"].map(document_order)

doc_sequence = doc_sequence.sort_values(
    ["captain_id", "event_ts"]
)

doc_sequence.head(20)

,event_id,captain_id,doc_type,attempt_no,event_type,event_ts,failure_reason,document_order
118462,EV000118462,CPT100000,DL,1,upload_success,2026-05-02 16:16:28.334434602,NaN,1
118463,EV000118463,CPT100000,DL,1,verification_pass,2026-05-02 22:37:15.678999608,NaN,1
118464,EV000118464,CPT100000,RC,1,upload_success,2026-05-03 06:23:32.362027418,NaN,2
118465,EV000118465,CPT100000,RC,1,verification_pass,2026-05-03 17:48:59.486740931,NaN,2
118466,EV000118466,CPT100000,AADHAAR,1,upload_success,2026-05-04 10:19:57.617946060,NaN,3
118467,EV000118467,CPT100000,AADHAAR,1,verification_pass,2026-05-04 12:16:11.389574146,NaN,3
118468,EV000118468,CPT100000,PERMIT,1,upload_success,2026-05-05 10:56:35.052084652,NaN,4
118469,EV000118469,CPT100000,PERMIT,1,verification_pass,2026-05-05 14:52:12.185964306,NaN,4
118470,EV000118470,CPT100000,FITNESS,1,upload_success,2026-05-05 23:13:33.449345027,NaN,5
118471,EV000118471,CPT100000,FITNESS,1,verification_fail,2026-05-06 13:51:36.735994908,ocr_low_confidence,5


In [16]:
first_doc_events = (
    doc_sequence
    .groupby(["captain_id", "doc_type"])["event_ts"]
    .min()
    .reset_index()
)

first_doc_events["document_order"] = (
    first_doc_events["doc_type"].map(document_order)
)

first_doc_events = first_doc_events.sort_values(
    ["captain_id", "event_ts"]
)

first_doc_events.head(20)

,captain_id,doc_type,event_ts,document_order
1,CPT100000,DL,2026-05-02 16:16:28.334434602,1
4,CPT100000,RC,2026-05-03 06:23:32.362027418,2
0,CPT100000,AADHAAR,2026-05-04 10:19:57.617946060,3
3,CPT100000,PERMIT,2026-05-05 10:56:35.052084652,4
2,CPT100000,FITNESS,2026-05-05 23:13:33.449345027,5
5,CPT100001,DL,2026-06-15 23:06:48.243606163,1
6,CPT100001,RC,2026-06-17 05:51:29.099685867,2
8,CPT100002,DL,2026-05-26 06:17:59.751491735,1
12,CPT100002,RC,2026-05-26 14:20:37.625804473,2
7,CPT100002,AADHAAR,2026-05-28 05:23:39.025174625,3


In [17]:
sequence_check = (
    first_doc_events
    .groupby("captain_id")["document_order"]
    .apply(list)
    .reset_index(name="observed_sequence")
)

sequence_check.head(20)

,captain_id,observed_sequence
0,CPT100000,"[1, 2, 3, 4, 5]"
1,CPT100001,"[1, 2]"
2,CPT100002,"[1, 2, 3, 4, 5, 6]"
3,CPT100003,"[1, 2]"
4,CPT100004,"[1, 2, 3, 5]"
5,CPT100005,"[1, 2]"
6,CPT100006,"[1, 2, 3, 4, 5, 6]"
7,CPT100007,"[1, 2]"
8,CPT100008,"[1, 2, 3, 4, 5, 6]"
9,CPT100009,"[1, 2]"


In [18]:
sequence_check["has_sequence_violation"] = sequence_check[
    "observed_sequence"
].apply(
    lambda x: x != sorted(x)
)

print(
    "Captains with document sequence violations:",
    sequence_check["has_sequence_violation"].sum()
)

Captains with document sequence violations: 0


## 3. Document Applicability Audit

Permit is required for Auto and Cab according to the onboarding process. I want to check whether Permit events appear only for captains whose vehicle type requires it, and whether Auto and Cab captains are expected to have Permit events.

In [19]:
permit_applicability = (
    doc_events[doc_events["doc_type"] == "PERMIT"]
    .merge(
        captains[["captain_id", "vehicle_type"]],
        on="captain_id",
        how="left"
    )
)

permit_applicability["permit_expected"] = (
    permit_applicability["vehicle_type"].isin(["Auto", "Cab"])
)

permit_applicability["permit_expected"].value_counts(dropna=False)

permit_expected
True    20807
Name: count, dtype: int64

In [20]:
invalid_permit_events = permit_applicability[
    ~permit_applicability["permit_expected"]
]

print(
    "Permit events for vehicles where Permit is not required:",
    len(invalid_permit_events)
)

Permit events for vehicles where Permit is not required: 0


In [21]:
eligible_captains = captains[
    captains["vehicle_type"].isin(["Auto", "Cab"])
][["captain_id", "vehicle_type"]]

captains_with_permit = set(
    permit_applicability["captain_id"].unique()
)

missing_permit = eligible_captains[
    ~eligible_captains["captain_id"].isin(captains_with_permit)
]

print(
    "Auto/Cab captains with no Permit event:",
    len(missing_permit)
)

Auto/Cab captains with no Permit event: 11031


## 4. Approval - Activation Consistency Audit

`activation` contains post-approval activity, so every activation record should correspond to a captain whose final onboarding status is `approved`.

I will also check whether approved captains without activation records exist, and whether any non-approved captains appear in the activation table.

In [22]:
approval_status = approvals[["captain_id", "final_status"]].copy()

activation_check = activation.merge(
    approval_status,
    on="captain_id",
    how="left"
)

activation_check["final_status"].value_counts(dropna=False)

final_status
approved    4206
Name: count, dtype: int64

In [23]:
invalid_activation = activation_check[
    activation_check["final_status"] != "approved"
]

print(
    "Activation records for non-approved captains:",
    len(invalid_activation)
)

Activation records for non-approved captains: 0


In [24]:
approved_captains = approvals[
    approvals["final_status"] == "approved"
]["captain_id"]

activation_captains = set(activation["captain_id"])

approved_without_activation = approved_captains[
    ~approved_captains.isin(activation_captains)
]

print(
    "Approved captains without activation record:",
    len(approved_without_activation)
)

Approved captains without activation record: 0


## 5. Cross-Table Temporal Consistency Audit

The captain lifecycle should follow a logical temporal order: signup → document processing → approval → first order.

I will check for impossible timestamp relationships across the captain-level lifecycle tables.

In [25]:
lifecycle = (
    captains[["captain_id", "signup_ts"]]
    .merge(
        approvals[["captain_id", "decision_ts", "final_status"]],
        on="captain_id",
        how="left"
    )
    .merge(
        activation[["captain_id", "first_order_ts"]],
        on="captain_id",
        how="left"
    )
)

In [26]:
approval_before_signup = lifecycle[
    lifecycle["decision_ts"].notna()
    & (lifecycle["decision_ts"] < lifecycle["signup_ts"])
]

print(
    "Approvals before signup:",
    len(approval_before_signup)
)

Approvals before signup: 0


In [27]:
order_before_signup = lifecycle[
    lifecycle["first_order_ts"].notna()
    & (lifecycle["first_order_ts"] < lifecycle["signup_ts"])
]

print(
    "First orders before signup:",
    len(order_before_signup)
)

First orders before signup: 0


In [28]:
order_before_approval = lifecycle[
    lifecycle["first_order_ts"].notna()
    & lifecycle["decision_ts"].notna()
    & (lifecycle["first_order_ts"] < lifecycle["decision_ts"])
]

print(
    "First orders before approval:",
    len(order_before_approval)
)

First orders before approval: 0


In [29]:
doc_after_order = (
    doc_events[["captain_id", "event_ts"]]
    .merge(
        activation[["captain_id", "first_order_ts"]],
        on="captain_id",
        how="inner"
    )
)

doc_after_order = doc_after_order[
    doc_after_order["first_order_ts"].notna()
    & (doc_after_order["event_ts"] > doc_after_order["first_order_ts"])
]

print(
    "Document events after first order:",
    len(doc_after_order)
)

Document events after first order: 0


## 6. Data Audit Summary

The raw datasets were audited for structural integrity, business-rule consistency, referential integrity, missingness, numeric validity, and lifecycle chronology before any cleaning or analytical transformations.

### Key findings

* All primary keys and expected identifiers were unique, and there were no full-row duplicates.
* Referential integrity checks passed across the captain-level datasets.
* No document events occurred before a captain's signup.
* Document attempts had no gaps, and no individual attempt contained both a verification failure and verification pass.
* Document processing followed the defined onboarding sequence. No sequence violations were observed.
* Permit events appeared only for Auto/Cab captains, as expected. However, 11,031 Auto/Cab captains had no Permit event, which is consistent with captains dropping out before reaching that stage and will be examined through the onboarding funnel rather than treated as a data error.
* All 4,206 activation records belonged to approved captains, and every approved captain had an activation record.
* No approvals occurred before signup, no first orders occurred before signup or approval, and no document events occurred after a captain's first order.
* Airport-hourly fulfillment arithmetic was internally consistent: fulfilled requests plus unfulfilled requests equaled total requests for every zone-hour record.
* Several data-quality issues were identified and will be retained and explicitly handled during processing rather than silently corrected:

  * 7 activation records have `orders_d7 > orders_d30`.
  * 457 nudge records have `clicked = 1` while `delivered = 0`.
  * `approvals.decision_ts` is missing for non-terminal onboarding statuses, while `last_stage_reached` has nulls for approved captains; these fields will therefore be interpreted according to their observed semantics rather than assumed to be universally populated.
  * Some D30 activation metrics are missing, reflecting incomplete observation windows and/or missing activity fields.

### Audit conclusion

The datasets are sufficiently structurally consistent to proceed with analysis. No rows will be removed solely because they represent incomplete onboarding journeys or missing downstream events. Data-quality anomalies will be documented and handled explicitly during the processing stage.

The next step is to create analysis-ready fields and datasets while preserving the raw data unchanged.
